## 1. 경로 / 설정

In [8]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# manifest 로드
manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
else:
    meta_lookup = {}

pdf_paths = sorted((DATA_RAW / "pdf").rglob("*.pdf"))
print(f"분석 대상 PDF: {len(pdf_paths)}개")
for p in pdf_paths:
    print(f"  {p.parent.name}/{p.name}")

분석 대상 PDF: 0개


## 2. 문서별 기본 정보 수집
PyMuPDF로 페이지 수, 파일 크기, 표/이미지 개수를 한 번에 추출한다.

In [ ]:
import fitz  # PyMuPDF
import pandas as pd


def inspect_pdf(pdf_path: Path) -> dict:
    """PDF 1개의 구조적 정보 추출."""
    doc = fitz.open(str(pdf_path))

    total_pages = len(doc)
    total_chars = 0
    total_images = 0
    total_tables = 0
    pages_with_text = 0

    for page in doc:
        text = page.get_text()
        if text.strip():
            pages_with_text += 1
        total_chars += len(text)
        total_images += len(page.get_images())
        # PyMuPDF의 find_tables는 1.23+ 버전 필요
        try:
            tables = page.find_tables()
            total_tables += len(tables.tables) if hasattr(tables, "tables") else 0
        except Exception:
            pass

    doc.close()

    extra = meta_lookup.get(pdf_path.name, {})
    return {
        "filename": pdf_path.name,
        "org": extra.get("org", pdf_path.parent.name),
        "language": extra.get("language", "?"),
        "doc_type": extra.get("doc_type", "?"),
        "file_size_mb": round(pdf_path.stat().st_size / 1024 / 1024, 2),
        "pages": total_pages,
        "pages_with_text": pages_with_text,
        "total_chars": total_chars,
        "avg_chars_per_page": round(total_chars / max(total_pages, 1)),
        "images": total_images,
        "tables": total_tables,
    }


rows = []
for p in pdf_paths:
    try:
        rows.append(inspect_pdf(p))
        print(f"  OK   {p.name}")
    except Exception as e:
        print(f"  FAIL {p.name}: {e}")

df_basic = pd.DataFrame(rows)
df_basic

## 한국어 vs 영어 문서 요약

In [ ]:
summary_by_lang = (
    df_basic.groupby("language")
    .agg(
        문서수=("filename", "count"),
        총페이지=("pages", "sum"),
        평균페이지=("pages", "mean"),
        평균이미지수=("images", "mean"),
        평균표수=("tables", "mean"),
    )
    .round(1)
)
summary_by_lang

## 3. 페이지별 텍스트 추출 → 노이즈 패턴 진단
각 페이지에서 자주 반복되는 짧은 텍스트(머리말/꼬리말/페이지번호 등)를 자동으로 식별한다.

In [ ]:
import re
from collections import Counter


def extract_repeated_lines(pdf_path: Path, top_n: int = 10, min_pages: int = 3) -> list:
    """여러 페이지에 반복되는 짧은 줄을 찾는다 (머리말/꼬리말 후보)."""
    doc = fitz.open(str(pdf_path))
    line_counter = Counter()

    for page in doc:
        text = page.get_text()
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        # 너무 긴 줄은 본문일 가능성 → 제외 (50자 이하만)
        short_lines = set(l for l in lines if 1 <= len(l) <= 50)
        for l in short_lines:
            line_counter[l] += 1

    doc.close()
    # min_pages 이상 등장한 줄만
    repeated = [
        (text, cnt)
        for text, cnt in line_counter.most_common(top_n * 3)
        if cnt >= min_pages
    ]
    return repeated[:top_n]


# 샘플 1개 PDF로 확인
sample = pdf_paths[0]
print(f"--- {sample.name} 반복 줄 top 10 ---")
for text, cnt in extract_repeated_lines(sample):
    print(f"  [{cnt}회] {text!r}")

## 페이지 번호 패턴 자동 감지

In [ ]:
PAGE_NUM_PATTERNS = [
    re.compile(r"^\d{1,4}$"),  # 그냥 숫자만
    re.compile(r"^- ?\d{1,4} ?-$"),  # -12-
    re.compile(r"^Page \d+", re.IGNORECASE),  # Page 12
    re.compile(r"^\d+\s*/\s*\d+$"),  # 12 / 24
]


def detect_noise_patterns(pdf_path: Path) -> dict:
    """PDF에서 발견된 노이즈 패턴 카운트."""
    doc = fitz.open(str(pdf_path))
    page_num_hits = 0
    repeated_short = 0
    line_counter = Counter()

    for page in doc:
        text = page.get_text()
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        for l in lines:
            if any(p.match(l) for p in PAGE_NUM_PATTERNS):
                page_num_hits += 1
            if 1 <= len(l) <= 50:
                line_counter[l] += 1

    doc.close()
    repeated_short = sum(1 for cnt in line_counter.values() if cnt >= 3)
    return {
        "page_number_hits": page_num_hits,
        "repeated_short_lines": repeated_short,
    }


noise_rows = []
for p in pdf_paths:
    try:
        d = detect_noise_patterns(p)
        d["filename"] = p.name
        noise_rows.append(d)
    except Exception as e:
        print(f"FAIL {p.name}: {e}")

df_noise = pd.DataFrame(noise_rows)[
    ["filename", "page_number_hits", "repeated_short_lines"]
]
df_basic_with_noise = df_basic.merge(df_noise, on="filename", how="left")
df_basic_with_noise

## 4. 페이지 단위 텍스트 → 토큰 길이 분포
tiktoken으로 페이지별 토큰 수를 계산한다.
이 분포가 chunk_size를 정하는 핵심 근거가 된다.

In [ ]:
import tiktoken

# tiktoken은 OpenAI tokenizer지만, 일반적인 토큰 길이 추정용으로 표준처럼 쓰임
enc = tiktoken.get_encoding("cl100k_base")


def page_token_lengths(pdf_path: Path) -> list[int]:
    doc = fitz.open(str(pdf_path))
    lens = []
    for page in doc:
        text = page.get_text()
        lens.append(len(enc.encode(text)))
    doc.close()
    return lens


page_token_rows = []
for p in pdf_paths:
    try:
        lens = page_token_lengths(p)
        extra = meta_lookup.get(p.name, {})
        for i, n in enumerate(lens):
            page_token_rows.append(
                {
                    "filename": p.name,
                    "org": extra.get("org", p.parent.name),
                    "language": extra.get("language", "?"),
                    "page": i + 1,
                    "tokens": n,
                }
            )
    except Exception as e:
        print(f"FAIL {p.name}: {e}")

df_tokens = pd.DataFrame(page_token_rows)
print(f"전체 페이지 수: {len(df_tokens)}")
print(df_tokens.groupby("language")["tokens"].describe().round(1))

## 시각화 1 — 페이지당 토큰 수 분포 (히스토그램)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정 (Windows / Mac 둘 다 시도)
for font in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
    if any(font in f.name for f in matplotlib.font_manager.fontManager.ttflist):
        matplotlib.rcParams["font.family"] = font
        break
matplotlib.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# (좌) 전체 분포
axes[0].hist(df_tokens["tokens"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(512, color="red", linestyle="--", label="chunk_size=512")
axes[0].axvline(1000, color="orange", linestyle="--", label="chunk_size=1000")
axes[0].set_xlabel("페이지당 토큰 수")
axes[0].set_ylabel("페이지 수")
axes[0].set_title("전체 문서 페이지별 토큰 분포")
axes[0].legend()

# (우) 언어별 비교
for lang, color in [("ko", "crimson"), ("en", "navy")]:
    sub = df_tokens[df_tokens["language"] == lang]["tokens"]
    if len(sub):
        axes[1].hist(
            sub, bins=30, alpha=0.5, label=f"{lang} (n={len(sub)})", color=color
        )
axes[1].set_xlabel("페이지당 토큰 수")
axes[1].set_ylabel("페이지 수")
axes[1].set_title("한국어 vs 영어 비교")
axes[1].legend()

plt.tight_layout()
plt.savefig(
    DATA_PROCESSED / "week4_token_distribution.png", dpi=120, bbox_inches="tight"
)
plt.show()

## 시각화 2 — 문서별 페이지 토큰 박스플롯

In [ ]:
# 파일별 박스플롯 (긴 파일명은 줄임)
fig, ax = plt.subplots(figsize=(12, max(4, len(pdf_paths) * 0.4)))

data = []
labels = []
for fn, sub in df_tokens.groupby("filename"):
    data.append(sub["tokens"].tolist())
    short = fn if len(fn) <= 40 else fn[:37] + "..."
    labels.append(short)

ax.boxplot(data, labels=labels, vert=False)
ax.axvline(512, color="red", linestyle="--", alpha=0.7, label="chunk_size=512")
ax.set_xlabel("페이지당 토큰 수")
ax.set_title("문서별 페이지 토큰 분포")
ax.legend()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / "week4_token_boxplot.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. 구조적 특성 진단 (수동 확인용 샘플)

In [ ]:
def find_image_heavy_pages(pdf_path: Path, top_n: int = 3) -> list:
    doc = fitz.open(str(pdf_path))
    page_info = []
    for i, page in enumerate(doc):
        img_count = len(page.get_images())
        text_len = len(page.get_text())
        page_info.append((i + 1, img_count, text_len))
    doc.close()
    page_info.sort(key=lambda x: x[1], reverse=True)
    return page_info[:top_n]


print("각 PDF의 이미지가 많은 페이지 top 3:")
for p in pdf_paths:
    heavy = find_image_heavy_pages(p)
    if heavy[0][1] > 0:
        print(f"\n{p.name}")
        for pg, imgs, chars in heavy:
            print(f"  p.{pg}: 이미지 {imgs}개 / 텍스트 {chars}자")

## 6. 진단 요약 표 (retrospective용)
이 표를 그대로 docs/week4_retrospective.md에 옮기면 된다.

In [ ]:
# 최종 요약 테이블
summary = df_basic_with_noise.copy()

# 페이지당 평균 토큰 추가
tok_avg = (
    df_tokens.groupby("filename")["tokens"]
    .agg(["mean", "median", "max"])
    .round(0)
    .astype(int)
)
tok_avg.columns = ["avg_tokens_per_page", "median_tokens", "max_tokens"]
summary = summary.merge(tok_avg, on="filename", how="left")

# 노이즈 비율
summary["page_number_ratio"] = (summary["page_number_hits"] / summary["pages"]).round(2)

summary_cols = [
    "filename",
    "org",
    "language",
    "pages",
    "avg_tokens_per_page",
    "median_tokens",
    "max_tokens",
    "images",
    "tables",
    "page_number_hits",
    "repeated_short_lines",
]
summary = summary[summary_cols]
summary.to_csv(
    DATA_PROCESSED / "week4_data_diagnosis.csv", index=False, encoding="utf-8-sig"
)
print(f"저장: {DATA_PROCESSED / 'week4_data_diagnosis.csv'}")
summary
